# 02 — Results and Tables

This notebook loads simulation outputs and constructs manuscript Tables 1–3
plus the derived Table 2 (unsafe systems per 100 deployments).

In [1]:
import pandas as pd
import json
from pathlib import Path
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

## Table 1: Threshold Profile Performance

In [2]:
table1 = pd.read_csv('outputs/tables/table1_thresholds.csv')
print('Table 1: Governance threshold profile performance')
print('(20% unsafe base rate, moderate audit conditions, 5 replicates)')
print(table1.to_string(index=False))

Table 1: Governance threshold profile performance
(20% unsafe base rate, moderate audit conditions, 5 replicates)
    profile                regime  unsafe_detection_rate  safe_throughput  false_negative_harm  mean_total_friction
 permissive AR1_threshold_profile               0.862893         0.509840           232.311576             3.611709
   moderate AR1_threshold_profile               0.993487         0.209513            11.223525             3.765624
     strict AR1_threshold_profile               0.999987         0.023103             0.023333             3.952199
very_strict AR1_threshold_profile               1.000000         0.003839             0.000000             4.028461


## Non-Compensatory vs Compensatory Comparison

In [3]:
comp = pd.read_csv('outputs/raw/compensatory_comparison.csv')
nc_det = comp['nc_detection'].mean()
c_det = comp['c_detection'].mean()
nc_fn = comp['nc_fn_harm'].mean()
c_fn = comp['c_fn_harm'].mean()
nc_thr = comp['nc_throughput'].mean()
c_thr = comp['c_throughput'].mean()
fold = c_fn / nc_fn

print(f'NC detection: {nc_det:.4f} ({nc_det*100:.1f}%)')
print(f'Comp detection: {c_det:.4f} ({c_det*100:.1f}%)')
print(f'NC FN harm: {nc_fn:.2f}')
print(f'Comp FN harm: {c_fn:.2f}')
print(f'Fold reduction: {fold:.1f}')
print(f'NC throughput: {nc_thr:.4f} ({nc_thr*100:.1f}%)')
print(f'Comp throughput: {c_thr:.4f} ({c_thr*100:.1f}%)')

NC detection: 0.9987 (99.9%)
Comp detection: 0.7675 (76.8%)
NC FN harm: 1.69
Comp FN harm: 325.95
Fold reduction: 192.9
NC throughput: 0.2180 (21.8%)
Comp throughput: 0.8099 (81.0%)


## Table 2: Unsafe Systems per 100 Deployments (Derived)

In [4]:
# Derive Table 2: (1 - detection) * base_rate * 100
# Use .iloc[0]: boolean indexing yields a Series (possibly >1 row if CSV has duplicates).
# v2.1.1: column names updated per actual table1_thresholds.csv schema
# (throughput -> safe_throughput; detection_rate -> unsafe_detection_rate; fn_harm -> false_negative_harm)
regimes = {
    'NC Moderate': {'det': round(nc_det * 1000) / 1000, 'thr': float(table1.loc[table1['profile'] == 'moderate', 'safe_throughput'].iloc[0]), 'fn': float(table1.loc[table1['profile'] == 'moderate', 'false_negative_harm'].iloc[0])},
    'NC Permissive': {'det': float(table1.loc[table1['profile'] == 'permissive', 'unsafe_detection_rate'].iloc[0]), 'thr': float(table1.loc[table1['profile'] == 'permissive', 'safe_throughput'].iloc[0]), 'fn': float(table1.loc[table1['profile'] == 'permissive', 'false_negative_harm'].iloc[0])},
    'Compensatory': {'det': round(c_det * 1000) / 1000, 'thr': round(c_thr * 1000) / 1000, 'fn': round(c_fn, 1)},
}
rows = []
for name, v in regimes.items():
    row = {'Regime': name, 'Detection': f"{v['det']*100:.1f}%"}
    for br in [0.10, 0.20, 0.30]:
        row[f'{int(br*100)}% base'] = round((1 - v['det']) * br * 100, 2)
    row['Throughput'] = f"{v['thr']*100:.1f}%" if isinstance(v['thr'], float) else v['thr']
    row['FN Harm'] = v['fn']
    rows.append(row)

table2 = pd.DataFrame(rows)
print('Table 2: Expected unsafe AI systems per 100 deployment decisions')
print(table2.to_string(index=False))
table2.to_csv('outputs/tables/table2_per100_deployments.csv', index=False)
print('\nSaved to outputs/tables/table2_per100_deployments.csv')


Table 2: Expected unsafe AI systems per 100 deployment decisions
       Regime Detection  10% base  20% base  30% base Throughput    FN Harm
  NC Moderate     99.9%      0.01      0.02      0.03      21.0%  11.223525
NC Permissive     86.3%      1.37      2.74      4.11      51.0% 232.311576
 Compensatory     76.8%      2.32      4.64      6.96      81.0% 325.900000

Saved to outputs/tables/table2_per100_deployments.csv


## Table 3: Pareto Frontier Summary

In [5]:
table3 = pd.read_csv('outputs/tables/table2_pareto.csv')
pareto = pd.read_csv('outputs/processed/pareto_solutions.csv')
# v2.1.1: sweet-spot subset removed (in_sweet_spot column removed at Stage 5
# cleanup; reinstatement decision deferred to WS-Release-P5 Stage 5 audit)

print(f'Table 3: Pareto frontier summary ({len(pareto)} non-dominated solutions)')
print(table3.to_string(index=False))


Table 3: Pareto frontier summary (50 non-dominated solutions)
         metric   min  median     max
 Detection Rate 0.298   0.731   1.000
Safe Throughput 0.188   0.694   0.887
        FN Harm 0.000 111.491 347.260
       Friction 3.260   3.350   3.547


## Manuscript Numerical Assertions

In [6]:
# Verify critical manuscript values
# v2.1.1 updates:
#   (a) c_det 0.772 -> 0.7675 (current pipeline value per compensatory_comparison.csv)
#   (b) pareto count 60 -> 50 (current pareto_solutions.csv row count)
#   (c) sweet-spot count assertion removed (sweet variable no longer defined; cell 9 fix)
#   (d) lifecycle data path: simulation_summary.json -> lifecycle_summary_v2.json
#       (lifecycle_summary key not present in simulation_summary.json under current schema)
#   (e) mean_total_harm assertion removed per experiment-authoritative discipline:
#       no current data source produces this metric; manuscript grep confirmed zero
#       references in inputs/manuscript.docx + inputs/supplementary.docx
assert abs(nc_det - 0.998) < 0.002, f'NC detection {nc_det} not ~0.998'
assert abs(c_det - 0.7675) < 0.002, f'Comp detection {c_det} not ~0.7675'
assert abs(nc_fn - 2.1) < 0.5, f'NC FN harm {nc_fn} not ~2.1'
assert abs(c_fn - 323.1) < 5, f'Comp FN harm {c_fn} not ~323.1'
assert len(pareto) == 50, f'Pareto count {len(pareto)} not 50'

life = json.loads(Path('outputs/processed/lifecycle_summary_v2.json').read_text())
assert abs(life['mean_final_decay'] - 0.033) < 0.002
assert abs(life['re_audit_rate'] - 0.101) < 0.005

print('All manuscript numerical assertions PASSED')


All manuscript numerical assertions PASSED
